In [45]:
from langsmith import Client
from qdrant_client import QdrantClient
import instructor
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
import os
import openai
from dotenv import load_dotenv
load_dotenv()
from qdrant_client.models import Distance,VectorParams,PointStruct,Filter,FieldCondition,MatchValue,Prefetch,Document,models

In [30]:
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
collection_name = os.getenv("QDRANT_COLLECTION_NAME")


In [31]:
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY,timeout=60)

### Download an example refernce data point from Langsmith

In [6]:
langsmith_client=Client(api_key=os.environ['LANGCHAIN_API_KEY'])

datasets=langsmith_client.read_dataset(
    dataset_name="shopmart_eval_dataset"
)

In [7]:
datasets

Dataset(name='shopmart_eval_dataset', description='Datasets for evaluating Rag pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('fbca5ce2-cdbf-44b6-859c-27caa2777a1f'), created_at=datetime.datetime(2026, 11, 10, 14, 53, 57, 832561, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 11, 10, 14, 53, 57, 832561, tzinfo=TzInfo(0)), example_count=80, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Windows-11-10.0.26300-SP0', 'sdk_version': '0.10.1', 'runtime_version': '3.12.14', 'langchain_version': '1.3.12', 'py_implementation': 'CPython', 'langchain_core_version': '1.4.9'}})

In [11]:
output_data=list(langsmith_client.list_examples(dataset_id=datasets.id,limit=100))

In [52]:
reference_output=output_data[2].outputs
reference_output

{'parent_asin': ['B0B9SY7F4Q'],
 'ground_truth': 'Yes — the ZAMKOL Bluetooth speaker (ZK706 Pro) includes an 8000mAh battery that can act as an emergency charger (power bank), supports up to 24 hours playtime and true wireless stereo pairing (TWS) to pair two speakers for 100W stereo output.',
 'reference_context_ids': ['B0B9SY7F4Q'],
 'reference_description': ['ZAMKOL Bluetooth Speaker, 50W Loud Portable Speaker with Deep Bass, IPX7 Waterproof, 8000mAh Emergency Charger, 24H Playtime, RGB Light, Outdoor Wireless Speakers for Party, Travel🎵 【HD Sound Quality & Powerful Bass】 - The wireless bluetooth speaker integrated a purpose-built 81mm NdFeB driver and an extra subwoofer, coupled with unique BAMBU algorithm, which offering the super powerful bass and crystal clear treble. The speaker delivers incredible volume through a dual-precision stereo driver. No distortion even at maximum volume.🎵 【RGB Lighting Effect & Bluetooth 5.0】 - The ZK706 Pro speaker offers 6 different lighting effect

In [54]:
reference_input=output_data[2].inputs
reference_input

{'question': 'Is there a wireless speaker that doubles as a power bank and supports TWS pairing?'}

In [55]:
from rag.reterivalgeneration import rag_pipeline

output = rag_pipeline(question="Is there a wireless speaker that doubles as a power bank and supports TWS pairing?")

In [56]:
output

{'original_output': RAGGenerationResponse(answer='Yes, there are wireless speakers that double as power banks and support True Wireless Stereo (TWS) pairing. For example, the ZAMKOL Bluetooth Speaker (ID: B09YYCBYTR and B0B9SY7F4Q) features a 50W loud output, IPX7 waterproof rating, 8000mAh battery that can be used as an emergency charger (power bank), 24 hours playtime, and supports 100W True Wireless Stereo Pairing (TWS) for immersive stereo sound.', references=[RAGUsedContext(id='B09YYCBYTR', description='ZAMKOL Bluetooth Speaker, 50W Loud Portable Speaker with Deep Bass, IPX7 Waterproof, 8000mAh Emergency Charger, 24H Playtime, RGB Light, Outdoor Wireless Speakers for Party, Travel'), RAGUsedContext(id='B0B9SY7F4Q', description='ZAMKOL Bluetooth Speaker, 50W Loud Portable Speaker with Deep Bass, IPX7 Waterproof, 8000mAh Emergency Charger, 24H Playtime, RGB Light, Outdoor Wireless Speakers for Party, Travel')]),
 'answer': 'Yes, there are wireless speakers that double as power banks